In [ ]:
# LogDet MFMC / MFMC / InfoNCE comparison for CEAP Dataset
# Subject-dependent single split, tri-modal emotion recognition.
# Modalities: EDA, BVP, SKT signals.

import os
import time
import pickle
from datetime import timedelta

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import train_test_split

print("CEAP subject-dependent comparison: MFMC, InfoNCE, LogDet")
print("=" * 70)

def select_device(min_free_memory_mb=10000):
    """Select the CUDA GPU with the most free VRAM, or stop if none is suitable."""
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is available. Stop this notebook instead of running on CPU.")

    try:
        import subprocess
        query = [
            "nvidia-smi",
            "--query-gpu=index,memory.free,memory.total,memory.used",
            "--format=csv,noheader,nounits",
        ]
        output = subprocess.check_output(query, encoding="utf-8")
        gpu_stats = []
        for line in output.strip().splitlines():
            index, free_mem, total_mem, used_mem = [part.strip() for part in line.split(",")]
            gpu_stats.append({
                "index": int(index),
                "free": int(free_mem),
                "total": int(total_mem),
                "used": int(used_mem),
            })
    except Exception as exc:
        raise RuntimeError(f"Could not query GPU memory with nvidia-smi: {exc}") from exc

    if not gpu_stats:
        raise RuntimeError("No CUDA GPUs were reported by nvidia-smi.")

    suitable_gpus = [gpu for gpu in gpu_stats if gpu["free"] >= min_free_memory_mb]
    if not suitable_gpus:
        status = ", ".join(
            f"GPU {gpu['index']}: {gpu['free']} MB free / {gpu['total']} MB total"
            for gpu in gpu_stats
        )
        raise RuntimeError(
            f"No GPU has at least {min_free_memory_mb} MB free VRAM. Current status: {status}"
        )

    selected = max(suitable_gpus, key=lambda gpu: gpu["free"])
    device = torch.device(f"cuda:{selected['index']}")
    print(
        f"Using device: {device} "
        f"({selected['free']} MB free / {selected['total']} MB total, {selected['used']} MB used)"
    )
    return device

def set_random_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

# Path usage note: set TAFFC_MFMC_ROOT if this repository is cloned elsewhere.
PROJECT_ROOT = os.environ.get('TAFFC_MFMC_ROOT', '/home/zhengdeyang/TAFFC_MFMC')
BASE_PATH = f'{PROJECT_ROOT}/MFMC/CEAP'
DATA_DIR = f'{BASE_PATH}/CEAP_Processed'
RESULTS_DIR = f'{BASE_PATH}/MFMC_InfoNCE_LogDet_Comparison/results/subject_dep_single_split'

# Single subject-dependent split parameters.
TEST_SIZE = 0.20
RANDOM_SEED = 42

# Training parameters.
BATCH_SIZE = 200
TOTAL_ITERATIONS = 20001  # Uses range(1, TOTAL_ITERATIONS), so 20001 means 20000 updates.
EVAL_INTERVAL = 500
CURVE_RECORD_INTERVAL = 100
LOG_INTERVAL = 500

# Optimizer parameters.
LEARNING_RATE_ENCODER = 0.0003
LEARNING_RATE_CLASSIFIER = 0.0003
BETA1 = 0.9
BETA2 = 0.999

# Method parameters.
COV_BETA = 0.5
LOGDET_EPS = 1e-5
INFONCE_TEMPERATURE = 0.2

# Classification loss option.
USE_CLASS_BALANCING = False

METHOD_ORDER = ['MFMC', 'InfoNCE', 'LogDet']
METHOD_FILENAMES = {
    'MFMC': 'mfmc_results.pkl',
    'InfoNCE': 'infonce_results.pkl',
    'LogDet': 'logdet_results.pkl',
}

os.makedirs(RESULTS_DIR, exist_ok=True)
set_random_seed(RANDOM_SEED)

print("Configuration loaded successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Single split test size: {TEST_SIZE}")
print(f"Total iterations: {TOTAL_ITERATIONS - 1}")
print(f"Batch size: {BATCH_SIZE}")
print(f"InfoNCE temperature: {INFONCE_TEMPERATURE}")
print(f"LogDet eps: {LOGDET_EPS}")

In [ ]:
# =============================================================================
# LOSS FUNCTIONS AND PROJECTION HEADS
# =============================================================================

def adaptive_estimation(v_t, beta, square_term, i, detach_square=True):
    """Exponential moving average with bias correction."""
    if detach_square:
        square_term = square_term.detach()
    v_t = beta * v_t + (1 - beta) * square_term
    return v_t.detach(), (v_t / (1 - beta ** i))

def mfmc_trace_loss(x, y, track_cov, i, cov_beta=0.95):
    """Trace MFMC loss for one feature pair."""
    Rx = (x.T @ x) / x.shape[0]
    Ry = (y.T @ y) / y.shape[0]
    Pxy = (x.T @ y) / x.shape[0]

    eps = 1e-6
    Rx = Rx + torch.eye(Rx.shape[0], device=Rx.device) * eps
    Ry = Ry + torch.eye(Ry.shape[0], device=Ry.device) * eps

    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, i, detach_square=True)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, i, detach_square=True)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, i, detach_square=True)

    Rx_est_inv = torch.inverse(Rx_est)
    Ry_est_inv = torch.inverse(Ry_est)

    cost = -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T \
           - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T

    return track_cov, -torch.trace(cost)

def tri_modal_projection_loss(fe1, fe2, fe3, proj12, proj23, proj13, trackers, step, cov_beta=0.5):
    """Trace MFMC tri-modal projection loss."""
    concat_12 = torch.cat([fe1, fe2], dim=1)
    concat_23 = torch.cat([fe2, fe3], dim=1)
    concat_13 = torch.cat([fe1, fe3], dim=1)

    proj_12 = proj12(concat_12)
    proj_23 = proj23(concat_23)
    proj_13 = proj13(concat_13)

    trackers['track_1_23'], loss1 = mfmc_trace_loss(fe1, proj_23, trackers['track_1_23'], step, cov_beta)
    trackers['track_2_13'], loss2 = mfmc_trace_loss(fe2, proj_13, trackers['track_2_13'], step, cov_beta)
    trackers['track_3_12'], loss3 = mfmc_trace_loss(fe3, proj_12, trackers['track_3_12'], step, cov_beta)

    return trackers, loss1 + loss2 + loss3

def fmca_logdet_loss(x, y, track_cov, i, cov_beta=0.95, eps=1e-5):
    """Log-determinant MFMC loss for one feature pair."""
    x_centered = x - x.mean(dim=0, keepdim=True)
    y_centered = y - y.mean(dim=0, keepdim=True)

    batch_size = x.shape[0]
    Rx = (x_centered.T @ x_centered) / batch_size
    Ry = (y_centered.T @ y_centered) / batch_size
    Pxy = (x_centered.T @ y_centered) / batch_size

    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, i, detach_square=False)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, i, detach_square=False)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, i, detach_square=False)

    feature_dim = Rx_est.shape[0]
    identity = torch.eye(feature_dim, device=x.device)
    Rx_reg = Rx_est + eps * identity
    Ry_reg = Ry_est + eps * identity

    upper = torch.cat((Rx_reg, Pxy_est), dim=1)
    lower = torch.cat((Pxy_est.T, Ry_reg), dim=1)
    R_joint = torch.cat((upper, lower), dim=0)

    try:
        sign_joint, logdet_joint = torch.linalg.slogdet(R_joint)
        sign_x, logdet_x = torch.linalg.slogdet(Rx_reg)
        sign_y, logdet_y = torch.linalg.slogdet(Ry_reg)

        if sign_joint <= 0 or sign_x <= 0 or sign_y <= 0:
            Rx_reg = Rx_est + (eps * 10) * identity
            Ry_reg = Ry_est + (eps * 10) * identity
            upper = torch.cat((Rx_reg, Pxy_est), dim=1)
            lower = torch.cat((Pxy_est.T, Ry_reg), dim=1)
            R_joint = torch.cat((upper, lower), dim=0)
            _, logdet_joint = torch.linalg.slogdet(R_joint)
            _, logdet_x = torch.linalg.slogdet(Rx_reg)
            _, logdet_y = torch.linalg.slogdet(Ry_reg)

        loss = logdet_joint - logdet_x - logdet_y
    except Exception as exc:
        print(f"Error computing log-determinants at iteration {i}: {exc}")
        loss = torch.tensor(0.0, device=x.device, requires_grad=True)

    return track_cov, loss

def tri_modal_projection_logdet_loss(fe1, fe2, fe3, proj12, proj23, proj13, trackers, step, cov_beta=0.5, eps=1e-5):
    """LogDet MFMC tri-modal projection loss."""
    concat_12 = torch.cat([fe1, fe2], dim=1)
    concat_23 = torch.cat([fe2, fe3], dim=1)
    concat_13 = torch.cat([fe1, fe3], dim=1)

    proj_12 = proj12(concat_12)
    proj_23 = proj23(concat_23)
    proj_13 = proj13(concat_13)

    trackers['track_1_23'], loss1 = fmca_logdet_loss(fe1, proj_23, trackers['track_1_23'], step, cov_beta, eps)
    trackers['track_2_13'], loss2 = fmca_logdet_loss(fe2, proj_13, trackers['track_2_13'], step, cov_beta, eps)
    trackers['track_3_12'], loss3 = fmca_logdet_loss(fe3, proj_12, trackers['track_3_12'], step, cov_beta, eps)

    return trackers, loss1 + loss2 + loss3

def info_nce_loss(x, y, temperature=0.1):
    """InfoNCE loss for one feature pair."""
    batch_size = x.shape[0]
    x = F.normalize(x, p=2, dim=1)
    y = F.normalize(y, p=2, dim=1)
    sim_matrix = torch.matmul(x, y.T) / temperature
    labels = torch.arange(batch_size, device=x.device)
    return F.cross_entropy(sim_matrix, labels)

def symmetric_info_nce_loss(x, y, temperature=0.1):
    """Symmetric InfoNCE loss for one feature pair."""
    return (info_nce_loss(x, y, temperature) + info_nce_loss(y, x, temperature)) / 2.0

def tri_modal_info_nce_loss(fe1, fe2, fe3, proj12, proj23, proj13, temperature=0.1):
    """Tri-modal projection InfoNCE loss from the DEAP InfoNCE_HO implementation."""
    concat_12 = torch.cat([fe1, fe2], dim=1)
    concat_23 = torch.cat([fe2, fe3], dim=1)
    concat_13 = torch.cat([fe1, fe3], dim=1)

    proj_12 = proj12(concat_12)
    proj_23 = proj23(concat_23)
    proj_13 = proj13(concat_13)

    loss1 = symmetric_info_nce_loss(fe1, proj_23, temperature)
    loss2 = symmetric_info_nce_loss(fe2, proj_13, temperature)
    loss3 = symmetric_info_nce_loss(fe3, proj_12, temperature)

    return (loss1 + loss2 + loss3) / 3.0

class ProjectionHead(nn.Module):
    """2-layer MLP projection head: Linear(256->512) + BN + ReLU -> Linear(512->128) + BN."""
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return x

print("Loss functions and projection heads defined successfully")

In [ ]:
# =============================================================================
# NEURAL NETWORK ARCHITECTURES
# =============================================================================

class NETWORK_F_MLP(nn.Module):
    """Multi-layer perceptron for feature transformation."""
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super(NETWORK_F_MLP, self).__init__()
        self.dim = out_dim
        self.num_layers = num_layers

        self.fc_list = []
        self.bn_list = []
        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        for _ in range(self.num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        self.fc_list = nn.ModuleList(self.fc_list)
        self.bn_list = nn.ModuleList(self.bn_list)
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)

        for i in range(self.num_layers):
            x = self.fc_list[i](x)
            x = torch.relu(x)
            x = self.bn_list[i](x)

        x = self.fc_final(x)
        x = torch.sigmoid(x)
        return x

class Advanced1DCNN_channel(nn.Module):
    """Advanced 1D CNN for CEAP physiological signals."""
    def __init__(self, input_channels=1, num_classes=128, input_size=1280):
        super(Advanced1DCNN_channel, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        feat_size = input_size // (4 * 4 * 4 * 4)
        self.fc1 = nn.Sequential(
            nn.Linear(256 * feat_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
        )
        self.fc3 = nn.Linear(512, num_classes)
        self.MLP = NETWORK_F_MLP(
            input_dim=128 * input_channels,
            hidden_dim=4000,
            out_dim=num_classes,
            num_layers=1,
        )

    def forward(self, x):
        batch_size, channels = x.shape[0], x.shape[1]
        x = x.unsqueeze(2)
        x = x.flatten(0, 1)

        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)
        out = out.reshape(batch_size, channels, -1)
        out = out.flatten(-2, -1)
        out = self.MLP(out)
        return out

class ComplexClassifier(nn.Module):
    """Multi-layer classifier for emotion recognition."""
    def __init__(self, dim_features=128, num_classes=4):
        super(ComplexClassifier, self).__init__()
        self.fc1 = nn.Linear(dim_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x

print("Neural network architectures defined successfully")

In [ ]:
# =============================================================================
# DATA LOADING AND SINGLE SUBJECT-DEPENDENT SPLIT
# =============================================================================

print("Loading CEAP dataset")
print("Modalities: EDA, BVP, SKT")

try:
    subject = np.load(f'{DATA_DIR}/subject.npy')
    emotion_labels = np.load(f'{DATA_DIR}/emotion_labels.npy')
    eda_data = np.load(f'{DATA_DIR}/eda_data.npy')
    bvp_data = np.load(f'{DATA_DIR}/bvp_data.npy')
    skt_data = np.load(f'{DATA_DIR}/skt_data.npy')

    subject = torch.from_numpy(subject).long()
    emotion_labels = torch.from_numpy(emotion_labels).long()
    eda_data = torch.from_numpy(eda_data).float()
    bvp_data = torch.from_numpy(bvp_data).float()
    skt_data = torch.from_numpy(skt_data).float()

    print("Data loaded successfully")
    print(f"Total samples: {eda_data.shape[0]}")
    print(f"EDA shape: {eda_data.shape}")
    print(f"BVP shape: {bvp_data.shape}")
    print(f"SKT shape: {skt_data.shape}")
    print(f"Number of unique subjects: {len(torch.unique(subject))}")
    print(f"Number of emotion classes: {len(torch.unique(emotion_labels))}")

    quadrant_names = [
        "Low V-Low A (Sad)",
        "Low V-High A (Angry)",
        "High V-Low A (Calm)",
        "High V-High A (Happy)",
    ]
    class_counts = torch.bincount(emotion_labels)
    print("Emotion class distribution:")
    for i, (name, count) in enumerate(zip(quadrant_names, class_counts)):
        print(f"  Class {i} - {name}: {count} samples ({count / len(emotion_labels) * 100:.1f}%)")

    indices = np.arange(eda_data.shape[0])
    train_indices, test_indices = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        shuffle=True,
        stratify=emotion_labels.numpy(),
    )

    train_eda = eda_data[train_indices]
    test_eda = eda_data[test_indices]
    train_bvp = bvp_data[train_indices]
    test_bvp = bvp_data[test_indices]
    train_skt = skt_data[train_indices]
    test_skt = skt_data[test_indices]
    train_labels = emotion_labels[train_indices]
    test_labels = emotion_labels[test_indices]

    print("Single subject-dependent split created")
    print(f"Train samples: {len(train_indices)}")
    print(f"Test samples: {len(test_indices)}")

    data_loaded = True

except FileNotFoundError as exc:
    print(f"Error loading data: {exc}")
    data_loaded = False
except Exception as exc:
    print(f"Unexpected error: {exc}")
    data_loaded = False

In [ ]:
# =============================================================================
# TRAINING HELPERS
# =============================================================================

if data_loaded:
    device = select_device()
    experiment_results = {}

    def create_trackers(device, feature_dim=128):
        return {
            'track_1_23': {
                'Rx': torch.zeros(feature_dim, feature_dim, device=device),
                'Ry': torch.zeros(feature_dim, feature_dim, device=device),
                'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
            },
            'track_2_13': {
                'Rx': torch.zeros(feature_dim, feature_dim, device=device),
                'Ry': torch.zeros(feature_dim, feature_dim, device=device),
                'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
            },
            'track_3_12': {
                'Rx': torch.zeros(feature_dim, feature_dim, device=device),
                'Ry': torch.zeros(feature_dim, feature_dim, device=device),
                'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
            },
        }

    def create_models(device):
        NET_EDA = Advanced1DCNN_channel(
            input_channels=eda_data.shape[1], num_classes=128, input_size=eda_data.shape[2]
        ).to(device)
        NET_BVP = Advanced1DCNN_channel(
            input_channels=bvp_data.shape[1], num_classes=128, input_size=bvp_data.shape[2]
        ).to(device)
        NET_SKT = Advanced1DCNN_channel(
            input_channels=skt_data.shape[1], num_classes=128, input_size=skt_data.shape[2]
        ).to(device)

        proj_12 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
        proj_23 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
        proj_13 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)

        num_classes = len(torch.unique(emotion_labels))
        classifier = ComplexClassifier(dim_features=128, num_classes=num_classes).to(device)
        return NET_EDA, NET_BVP, NET_SKT, proj_12, proj_23, proj_13, classifier

    def evaluate_classifier(NET_EDA, classifier):
        NET_EDA.eval()
        classifier.eval()

        correct = 0
        total = 0
        test_batch_size = 100
        num_test_batches = (len(test_eda) + test_batch_size - 1) // test_batch_size

        with torch.no_grad():
            for batch_idx in range(num_test_batches):
                start_idx = batch_idx * test_batch_size
                end_idx = min((batch_idx + 1) * test_batch_size, len(test_eda))

                test_eda_batch = test_eda[start_idx:end_idx].to(device)
                test_labels_batch = test_labels[start_idx:end_idx].to(device)

                outputs = classifier(NET_EDA(test_eda_batch))
                _, predicted = torch.max(outputs.data, 1)
                total += test_labels_batch.size(0)
                correct += (predicted == test_labels_batch).sum().item()

        NET_EDA.train()
        classifier.train()
        return correct / total

    def compute_projection_cost(method_name, feature_eda, feature_bvp, feature_skt, proj_12, proj_23, proj_13, trackers, iteration):
        if method_name == 'MFMC':
            trackers, cost = tri_modal_projection_loss(
                feature_eda, feature_bvp, feature_skt,
                proj_12, proj_23, proj_13, trackers, iteration, COV_BETA,
            )
            return trackers, cost

        if method_name == 'InfoNCE':
            cost = tri_modal_info_nce_loss(
                feature_eda, feature_bvp, feature_skt,
                proj_12, proj_23, proj_13, INFONCE_TEMPERATURE,
            )
            return trackers, cost

        if method_name == 'LogDet':
            trackers, cost = tri_modal_projection_logdet_loss(
                feature_eda, feature_bvp, feature_skt,
                proj_12, proj_23, proj_13, trackers, iteration, COV_BETA, LOGDET_EPS,
            )
            return trackers, cost

        raise ValueError(f"Unknown method: {method_name}")

    def save_method_result(method_name, result):
        result_path = f"{RESULTS_DIR}/{METHOD_FILENAMES[method_name]}"
        with open(result_path, 'wb') as f:
            pickle.dump(result, f)
        print(f"Saved {method_name} result to {result_path}")

    def train_method(method_name):
        if method_name not in METHOD_ORDER:
            raise ValueError(f"method_name must be one of {METHOD_ORDER}")

        print("=" * 70)
        print(f"Training {method_name}")
        print("=" * 70)

        # Reset seed so all methods use comparable initialization and minibatch order.
        set_random_seed(RANDOM_SEED)

        NET_EDA, NET_BVP, NET_SKT, proj_12, proj_23, proj_13, classifier = create_models(device)
        trackers = create_trackers(device)

        all_feature_params = (
            list(NET_EDA.parameters()) + list(NET_BVP.parameters()) + list(NET_SKT.parameters())
            + list(proj_12.parameters()) + list(proj_23.parameters()) + list(proj_13.parameters())
        )
        optimizer_features = optim.Adam(
            all_feature_params,
            lr=LEARNING_RATE_ENCODER,
            betas=(BETA1, BETA2),
            amsgrad=True,
        )
        optimizer_classifier = optim.Adam(
            classifier.parameters(),
            lr=LEARNING_RATE_CLASSIFIER,
            betas=(BETA1, BETA2),
            amsgrad=True,
        )

        if USE_CLASS_BALANCING:
            train_class_counts = torch.bincount(train_labels)
            class_weights = 1.0 / train_class_counts.float()
            class_weights = class_weights / class_weights.sum() * len(class_weights)
            criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
        else:
            criterion = nn.CrossEntropyLoss()

        cost_history = []
        classifier_loss_history = []
        curve_iterations = []
        accuracy_history = []
        accuracy_iterations = []
        best_accuracy = 0.0
        best_accuracy_iteration = 0

        start_time = time.time()

        for iteration in range(1, TOTAL_ITERATIONS):
            # Phase 1: unsupervised encoder/projection training.
            optimizer_features.zero_grad()
            batch_indices = torch.randperm(len(train_eda))[:BATCH_SIZE]

            input_eda = train_eda[batch_indices].to(device)
            input_bvp = train_bvp[batch_indices].to(device)
            input_skt = train_skt[batch_indices].to(device)

            feature_eda = NET_EDA(input_eda)
            feature_bvp = NET_BVP(input_bvp)
            feature_skt = NET_SKT(input_skt)

            trackers, projection_cost = compute_projection_cost(
                method_name,
                feature_eda, feature_bvp, feature_skt,
                proj_12, proj_23, proj_13,
                trackers, iteration,
            )
            projection_cost.backward()
            optimizer_features.step()

            # Phase 2: supervised classifier training on detached EDA features.
            optimizer_classifier.zero_grad()
            batch_indices = torch.randperm(len(train_eda))[:BATCH_SIZE]
            input_eda = train_eda[batch_indices].to(device)
            labels_batch = train_labels[batch_indices].to(device)

            with torch.no_grad():
                feature_eda = NET_EDA(input_eda)

            output_class = classifier(feature_eda.detach())
            classifier_loss = criterion(output_class, labels_batch)
            classifier_loss.backward()
            optimizer_classifier.step()

            if iteration == 1 or iteration % CURVE_RECORD_INTERVAL == 0:
                cost_history.append(float(projection_cost.item()))
                classifier_loss_history.append(float(classifier_loss.item()))
                curve_iterations.append(iteration)

            if iteration % LOG_INTERVAL == 0:
                print(
                    f"{method_name} | Iter {iteration:5d} | "
                    f"Cost: {projection_cost.item():.6f} | "
                    f"CE: {classifier_loss.item():.6f}"
                )

            if iteration % EVAL_INTERVAL == 0:
                accuracy = evaluate_classifier(NET_EDA, classifier)
                accuracy_history.append(float(accuracy))
                accuracy_iterations.append(iteration)

                if accuracy > best_accuracy:
                    best_accuracy = float(accuracy)
                    best_accuracy_iteration = iteration

                print(f"{method_name} | Iter {iteration:5d} | Test accuracy: {accuracy:.4f}")

        training_time = time.time() - start_time
        result = {
            'method': method_name,
            'cost_history': cost_history,
            'classifier_loss_history': classifier_loss_history,
            'curve_iterations': curve_iterations,
            'accuracy_history': accuracy_history,
            'accuracy_iterations': accuracy_iterations,
            'best_accuracy': best_accuracy,
            'best_accuracy_iteration': best_accuracy_iteration,
            'final_accuracy': accuracy_history[-1] if accuracy_history else 0.0,
            'training_time': training_time,
            'config': {
                'test_size': TEST_SIZE,
                'random_seed': RANDOM_SEED,
                'batch_size': BATCH_SIZE,
                'total_iterations': TOTAL_ITERATIONS - 1,
                'eval_interval': EVAL_INTERVAL,
                'curve_record_interval': CURVE_RECORD_INTERVAL,
                'cov_beta': COV_BETA,
                'logdet_eps': LOGDET_EPS,
                'infonce_temperature': INFONCE_TEMPERATURE,
                'learning_rate_encoder': LEARNING_RATE_ENCODER,
                'learning_rate_classifier': LEARNING_RATE_CLASSIFIER,
                'use_class_balancing': USE_CLASS_BALANCING,
            },
        }

        save_method_result(method_name, result)
        print(f"{method_name} complete")
        print(f"Best accuracy: {best_accuracy:.4f} at iteration {best_accuracy_iteration}")
        print(f"Training time: {str(timedelta(seconds=int(training_time)))}")

        del NET_EDA, NET_BVP, NET_SKT, proj_12, proj_23, proj_13, classifier
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return result

    print("Training helpers ready")
else:
    print("Cannot initialize training helpers because data loading failed")

In [ ]:
# =============================================================================
# TRAIN MFMC TRACE VERSION
# =============================================================================

if data_loaded:
    experiment_results['MFMC'] = train_method('MFMC')
else:
    print("Cannot train MFMC because data loading failed")

In [ ]:
# =============================================================================
# TRAIN INFONCE VERSION
# =============================================================================

if data_loaded:
    experiment_results['InfoNCE'] = train_method('InfoNCE')
else:
    print("Cannot train InfoNCE because data loading failed")

In [ ]:
# =============================================================================
# TRAIN LOGDET MFMC VERSION
# =============================================================================

if data_loaded:
    experiment_results['LogDet'] = train_method('LogDet')
else:
    print("Cannot train LogDet because data loading failed")

In [ ]:
# =============================================================================
# COMPARISON VISUALIZATION
# =============================================================================

# This cell creates a 3x3 ablation-style figure like the DEAP example:
# rows are methods, columns are projection cost, classifier CE loss, and test accuracy.

def load_saved_method_results():
    loaded = {}
    for method_name in METHOD_ORDER:
        result_path = f"{RESULTS_DIR}/{METHOD_FILENAMES[method_name]}"
        try:
            with open(result_path, 'rb') as f:
                loaded[method_name] = pickle.load(f)
            print(f"Loaded {method_name}: {result_path}")
        except FileNotFoundError:
            print(f"Missing {method_name} result: {result_path}")
    return loaded

def get_results_for_plot():
    results = {}
    if 'experiment_results' in globals():
        results.update(experiment_results)

    missing = [method for method in METHOD_ORDER if method not in results]
    if missing:
        saved_results = load_saved_method_results()
        results.update(saved_results)

    return {method: results[method] for method in METHOD_ORDER if method in results}

def style_axis(ax):
    ax.set_facecolor('#EAEAF2')
    ax.grid(True, color='white', linewidth=1.6)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.3)
    ax.tick_params(labelsize=13, width=1.1)

def plot_method_comparison(results, save_base_path):
    missing = [method for method in METHOD_ORDER if method not in results]
    if missing:
        print(f"Cannot make full comparison plot. Missing results for: {missing}")
        return

    plt.rcParams.update({
        'font.family': 'serif',
        'axes.titlesize': 24,
        'axes.labelsize': 18,
        'xtick.labelsize': 13,
        'ytick.labelsize': 13,
        'figure.dpi': 120,
    })

    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    cost_titles = {
        'MFMC': 'MFMC Projection Cost',
        'InfoNCE': 'InfoNCE Projection Cost',
        'LogDet': 'LogDet Projection Cost',
    }

    for row, method_name in enumerate(METHOD_ORDER):
        result = results[method_name]
        curve_x = np.array(result['curve_iterations'], dtype=float) / 100.0
        acc_x = np.array(result['accuracy_iterations'], dtype=float) / 100.0

        cost_values = np.array(result['cost_history'], dtype=float)
        ce_values = np.array(result['classifier_loss_history'], dtype=float)
        acc_values = np.array(result['accuracy_history'], dtype=float)

        # Projection cost.
        ax = axes[row, 0]
        style_axis(ax)
        ax.plot(curve_x, cost_values, color='#1f77b4', linewidth=2.8)
        ax.set_title(cost_titles[method_name], pad=14)
        ax.set_xlabel('Iteration (x100)')
        ax.set_ylabel('Cost')
        ax.set_xlim(left=0)

        # Classifier cross-entropy loss.
        ax = axes[row, 1]
        style_axis(ax)
        ax.plot(curve_x, ce_values, color='#ff7f0e', linewidth=2.8)
        ax.set_title('Classifier CE Loss', pad=14)
        ax.set_xlabel('Iteration (x100)')
        ax.set_ylabel('Loss')
        ax.set_xlim(left=0)

        # Test accuracy.
        ax = axes[row, 2]
        style_axis(ax)
        if len(acc_values) > 0:
            ax.plot(acc_x, acc_values, color='#d62728', linewidth=2.8)
            best_idx = int(np.argmax(acc_values))
            best_x = acc_x[best_idx]
            best_y = acc_values[best_idx]
            ax.scatter([best_x], [best_y], s=130, color='black', zorder=5)
            ax.annotate(
                f"{best_y * 100:.1f}%",
                xy=(best_x, best_y),
                xytext=(0, 12),
                textcoords='offset points',
                ha='center',
                fontsize=18,
                color='black',
            )
            if method_name == 'LogDet':
                ax.vlines(best_x, 0.0, best_y, color='#d62728', linewidth=2.3)
        ax.set_title('Test Accuracy', pad=14)
        ax.set_xlabel('Iteration (x100)')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0.0, 1.0)
        ax.set_xlim(left=0)

    fig.tight_layout(h_pad=2.0, w_pad=2.0)
    png_path = f'{save_base_path}.png'
    pdf_path = f'{save_base_path}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Comparison figure saved to: {png_path}')
    print(f'Comparison figure saved to: {pdf_path}')

plot_results = get_results_for_plot()
plot_method_comparison(plot_results, f'{RESULTS_DIR}/ceap_ablation_curves')

In [ ]:
# =============================================================================
# SUMMARY TABLE
# =============================================================================

plot_results = get_results_for_plot()
if plot_results:
    print("Method summary")
    print("=" * 70)
    for method_name in METHOD_ORDER:
        if method_name not in plot_results:
            continue
        result = plot_results[method_name]
        print(
            f"{method_name:7s} | "
            f"Best Acc: {result['best_accuracy']:.4f} | "
            f"Final Acc: {result['final_accuracy']:.4f} | "
            f"Best Iter: {result['best_accuracy_iteration']} | "
            f"Time: {str(timedelta(seconds=int(result['training_time'])))}"
        )
else:
    print("No results available yet")